# Notebook 1 — Introduction to Feature Engineering
**Sprint 6 · AI/ML Engineer Track**
**Dataset:** Telecom Customer Churn (synthetic, realistic) — `data/telecom_customers.csv`
**Target:** `churn` (Yes/No)

This notebook lays the conceptual foundation before any code is written. As a senior
engineer, I treat this stage the same way I'd treat a design doc before writing
production code — get the mental model right, and the implementation becomes obvious.

## 1. What is a Feature?

A **feature** is any measurable input variable that a Machine Learning model uses to
learn a pattern. It's the model's *only* window into the real world — if information
isn't encoded as a feature, the model literally cannot see it, no matter how powerful
the algorithm is.

**Feature vs Variable:** every feature is a variable, but not every variable is a
feature. A variable is any column in raw data. A feature is a variable (or a
transformation of one/many variables) that has been deliberately shaped to carry
predictive signal to the model. `customer_id` is a variable — it is *not* a feature
(it carries no generalizable signal and would just be memorized).

**Feature vs Target:** the target (`churn`) is what we want to predict. Features are
what we're allowed to know *before* the outcome happens. Confusing this — using
information that only exists *after* the target is known — is called **leakage**,
covered in depth in Notebook 12.

## 2. What is Feature Engineering, and Why Does It Matter?

Feature Engineering is the process of using domain knowledge and data transformation
techniques to create new input variables (or reshape existing ones) that make the
underlying pattern easier for a model to learn.

**Real-world example:** A telecom churn model with only `signup_date` and `today's date`
as raw columns has nothing to learn from — those dates are effectively unique per
customer. But `tenure_months = (today - signup_date)` turns that into a powerful,
generalizable signal: "customers with < 6 months tenure churn far more."

**Business example:** In banking fraud detection, the raw `transaction_timestamp` is
almost useless. But `time_since_last_transaction` and `transaction_amount / average
_monthly_spend` are exactly the kind of ratio/velocity features fraud teams rely on.

**AI/ML use case:** In this sprint's dataset, `monthly_charges` alone is informative,
but `monthly_charges relative to tenure`, or `total_charges / tenure_months` (an
implied "expected vs actual" ratio), often carries *more* signal than either raw
column alone — this is the core idea behind interaction features (Notebook 6).

> **Senior engineer's rule of thumb:** a good algorithm on poor features will almost
> always lose to a simple algorithm (e.g. Logistic Regression) on well-engineered
> features. Feature quality has a higher ceiling on model performance than algorithm
> choice, in the majority of tabular ML problems.

## 3. Importance of Domain Knowledge

Feature Engineering is not a purely mechanical or statistical exercise — the best
features come from understanding *how the business actually works*.

- A **data scientist without domain knowledge** might create `tenure_months` and stop.
- A **domain-aware engineer** knows that telecom churn is driven by *contract lock-in*,
  so they also engineer `is_month_to_month`, `months_until_contract_renewal`, and
  `payment_method_is_manual` (manual payments correlate with higher churn — customers
  who haven't set up autopay are less "sticky").

This is why Step 4 of this sprint ("Justify Your Feature") forces you to write down
*business meaning*, not just code — a feature you can't explain in one sentence to a
product manager is a feature you probably shouldn't ship.

## 4. Good Features vs Bad Features

| Good Feature | Bad Feature |
|---|---|
| Generalizes across many rows (e.g. `tenure_months`) | Unique per row (e.g. `customer_id`) |
| Available *before* the prediction is needed | Only known *after* the outcome (leakage) |
| Has a clear business interpretation | Arbitrary transformation with no interpretation |
| Stable over time / across train & test | Derived from a statistic computed on the full dataset (including test rows) |
| Reasonable cardinality if categorical | Extremely high-cardinality categorical dumped into one-hot (e.g. raw ZIP code with no grouping) |

## 5. The Feature Engineering Workflow

```
Raw Data
   │
   ▼
Feature Creation        → build new columns from existing ones (ratios, dates, text stats)
   │
   ▼
Feature Transformation  → reshape distributions (log, scaling, encoding)
   │
   ▼
Feature Extraction      → derive compact representations (PCA, TF-IDF)
   │
   ▼
Feature Selection       → keep only what actually helps the model
   │
   ▼
ML-Ready Features
```

- **Feature Creation**: deriving `tenure_months` from `signup_date`.
- **Feature Transformation**: log-transforming a skewed `total_charges` column.
- **Feature Extraction**: converting `support_ticket_text` into TF-IDF vectors, or
  compressing 20 correlated numeric columns into 3 principal components.
- **Feature Selection**: dropping a feature that turned out to be 99% correlated with
  another, or has near-zero variance.
- **Feature Leakage**: the failure mode that can silently invalidate all of the above —
  using information the model would not have access to at prediction time.

In [ ]:
import pandas as pd
import numpy as np

customers = pd.read_csv("./telecom_customers.csv", parse_dates=["signup_date"])
print("Shape:", customers.shape)
customers.info()

In [ ]:
customers.head()

## 6. Role of Feature Engineering in the ML Pipeline

Feature Engineering sits *between* data cleaning and modeling, and it is where most of
the "engineering judgment" in a real ML project actually lives:

```
[Sprint 5]                [Sprint 6 — this sprint]              [Sprint 7]
Raw Data → Clean Data →  Feature Engineering → Feature Selection → ML-Ready Features → Model Training
```

A senior engineer's mental checklist at this stage, applied to every candidate feature:
1. **Why** does this feature exist — what real-world signal is it capturing?
2. **When** would this information actually be available in production?
3. **Could** this feature leak future/target information?
4. **How** does it change the distribution / cardinality the model has to deal with?
5. **Is it worth the complexity** — does it measurably improve the model, or just add
   noise and maintenance cost?

The rest of this sprint builds the technique library needed to answer these questions
concretely rather than by intuition alone.